# Steam Oyun Benzerlik Ağı – Analiz, Bölüm 1
Beril Özdizdar · Sosyal Ağ Analizi dönem ödevi

Bu defter: temel ağ ölçüleri, derece dağılımı, merkezilik, topluluk tespiti (Louvain ve Leiden) ve null model karşılaştırmasını yapar.

**Nasıl çalıştırılır:** Hücreleri sırayla, yukarıdan aşağı çalıştır (hücreye tıkla, Shift+Enter). Her hücrenin çıktısını bekle, sonra sonrakine geç.
Her bölümün altındaki **Yorum** hücrelerine sonuçları **kendi cümlelerinle** yazacaksın (sunumda bunu soracaklar).

## 0. Kurulum

In [ ]:
!pip -q install igraph leidenalg powerlaw

### Veriyi yükle (zip'i bu hücre sorar)

In [ ]:
import os, glob, subprocess
from google.colab import files

if not glob.glob("/content/**/processed/edges_game_tag.csv", recursive=True):
    if not glob.glob("/content/steam-network*.zip"):
        print("Açılan pencereden steam-network.zip dosyasını seç:")
        files.upload()
    for z in glob.glob("/content/steam-network*.zip"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", "/content"])

hit = glob.glob("/content/**/processed/edges_game_tag.csv", recursive=True)
assert hit, "Veri bulunamadı: zip'in içinde data/processed klasörü olduğundan emin ol."
BASE = os.path.dirname(os.path.dirname(hit[0]))
print("Veri bulundu:", BASE)

In [ ]:
import os, json, random, warnings
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import normalized_mutual_info_score as nmi, adjusted_rand_score as ari
warnings.filterwarnings("ignore")

SEED = 42                      # sabit rastgele tohum
random.seed(SEED); np.random.seed(SEED)

OUT = "/content/sonuclar"              # sonuçlar buraya kaydedilir
os.makedirs(OUT, exist_ok=True)

## 1. Veriyi yükle ve oyun–oyun ağını kur

In [ ]:
gt = pd.read_csv(f"{BASE}/processed/edges_game_tag.csv", encoding="utf-8-sig")
gg = pd.read_csv(f"{BASE}/processed/edges_game_game.csv", encoding="utf-8-sig")

meta = {}
for line in open(f"{BASE}/raw/appdetails.jsonl", encoding="utf-8"):
    if line.strip():
        d = json.loads(line)
        meta[d["appid"]] = {"oyun": d.get("name", ""),
                            "degerlendirme": d.get("positive", 0) + d.get("negative", 0)}

G = nx.Graph()
G.add_nodes_from(gt["appid"].unique())          # izole oyunlar da ağda kalsın
G.add_weighted_edges_from(gg[["appid_a", "appid_b", "benzerlik"]].itertuples(index=False, name=None))
for n in G.nodes():
    G.nodes[n]["oyun"] = meta.get(n, {}).get("oyun", str(n))
print("Düğüm:", G.number_of_nodes(), "| Kenar:", G.number_of_edges())

## 2. Temel ağ ölçüleri

In [ ]:
def lcc(H):
    return H.subgraph(max(nx.connected_components(H), key=len)).copy()

def sampled_paths(H, k=300, seed=SEED):
    # Büyük ağda örneklemeli: k kaynak düğümden BFS
    rng = random.Random(seed)
    srcs = rng.sample(list(H.nodes()), min(k, H.number_of_nodes()))
    total = cnt = ecc = 0
    for s in srcs:
        dist = nx.single_source_shortest_path_length(H, s)
        total += sum(dist.values()); cnt += len(dist) - 1
        ecc = max(ecc, max(dist.values()))
    return total / cnt, ecc

H = lcc(G)   # en büyük bağlı bileşen
avg_path, diam_lb = sampled_paths(H)
comps = list(nx.connected_components(G))

olcu = {
    "Düğüm sayısı": G.number_of_nodes(),
    "Kenar sayısı": G.number_of_edges(),
    "Yoğunluk": round(nx.density(G), 5),
    "Bileşen sayısı": len(comps),
    "En büyük bileşen (düğüm)": H.number_of_nodes(),
    "İzole düğüm": sum(1 for _, d in G.degree() if d == 0),
    "Ortalama kümelenme": round(nx.average_clustering(G), 4),
    "Global kümelenme (transitivite)": round(nx.transitivity(G), 4),
    "Derece benzeşimi (assortativity)": round(nx.degree_assortativity_coefficient(G), 4),
    "Ortalama en kısa yol (en büyük bileşen, örneklemeli)": round(avg_path, 3),
    "Çap (alt sınır, örneklemeli)": diam_lb,
}
olcu_df = pd.DataFrame({"değer": olcu})
olcu_df.to_csv(f"{OUT}/temel_olculer.csv", encoding="utf-8-sig")
olcu_df

**Yorum (kendi cümlelerinle yaz):** Her satır için "bu sayı benim ağımda ne demek?" cümlesi.
- Yoğunluk: …
- Bileşenler / izole oyunlar: …
- Kümelenme: …
- Derece benzeşimi: …
- Ortalama yol ve çap: …

## 3. Derece dağılımı (log-log) ve kuvvet yasası testi

In [ ]:
deg = np.array([d for _, d in G.degree() if d > 0])
vals, counts = np.unique(deg, return_counts=True)
sd = np.sort(deg); ccdf = 1 - np.arange(len(sd)) / len(sd)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(vals, counts / counts.sum(), "o", ms=4)
ax[0].set(xlabel="derece k", ylabel="P(k)", title="Derece dağılımı (log-log)")
ax[1].loglog(sd, ccdf, ".")
ax[1].set(xlabel="derece k", ylabel="P(K ≥ k)", title="Tamamlayıcı birikimli dağılım (CCDF)")
plt.tight_layout(); plt.savefig(f"{OUT}/derece_dagilimi.png", dpi=150); plt.show()

In [ ]:
import powerlaw
fit = powerlaw.Fit(deg, discrete=True, verbose=False)
print(f"Kuvvet yasası: alpha = {fit.power_law.alpha:.3f}, xmin = {fit.xmin}, sigma = {fit.power_law.sigma:.3f}")
rows = []
for alt in ["lognormal", "exponential", "truncated_power_law"]:
    R, p = fit.distribution_compare("power_law", alt, normalized_ratio=True)
    kazanan = "kuvvet yasası" if R > 0 else alt
    rows.append({"alternatif": alt, "R": round(R, 3), "p": round(p, 4), "daha iyi uyan (işarete göre)": kazanan})
pl_df = pd.DataFrame(rows)
pl_df.to_csv(f"{OUT}/kuvvet_yasasi_testi.csv", index=False, encoding="utf-8-sig")
pl_df

**Yorum:** R > 0 ise kuvvet yasası, R < 0 ise alternatif dağılım daha iyi uyuyor. p küçükse (< 0.1) fark anlamlı.
Derece dağılımı kuvvet yasasına uyuyor mu? Uymuyorsa bu ağın yapısı için ne anlama gelir? …

## 4. Merkeziyet ölçüleri

In [ ]:
# Not: kenar ağırlıkları benzerlik olduğu için betweenness/closeness ağırlıksız (hop sayısı) hesaplanır.
print("Betweenness hesaplanıyor (örneklemeli, k=300)...")
btw = nx.betweenness_centrality(H, k=min(300, H.number_of_nodes()), seed=SEED)
print("Closeness hesaplanıyor (1-3 dakika sürebilir)...")
clo = nx.closeness_centrality(H)
deg_c = nx.degree_centrality(G)
pr = nx.pagerank(G, weight="weight")

cen = pd.DataFrame({
    "oyun": [G.nodes[n]["oyun"] for n in G.nodes()],
    "derece": [deg_c[n] for n in G.nodes()],
    "betweenness": [btw.get(n, 0.0) for n in G.nodes()],
    "closeness": [clo.get(n, 0.0) for n in G.nodes()],
    "pagerank": [pr[n] for n in G.nodes()],
}, index=list(G.nodes()))
cen.index.name = "appid"
cen.to_csv(f"{OUT}/merkeziyet.csv", encoding="utf-8-sig")

for col in ["derece", "betweenness", "closeness", "pagerank"]:
    print(f"\n=== İlk 10: {col} ===")
    print(cen[col].nlargest(10).to_frame().join(cen["oyun"]).to_string())

In [ ]:
spear = cen[["derece", "betweenness", "closeness", "pagerank"]].corr(method="spearman").round(3)
spear.to_csv(f"{OUT}/merkeziyet_spearman.csv", encoding="utf-8-sig")
spear

**Yorum:** Her ölçü için ilk 10'a bak.
- Derece yüksek olan oyun ne demek (çok oyunla benzer etiketler)? …
- Betweenness yüksek olan oyun ağda neyi bağlıyor (hangi iki tür arasında köprü)? …
- Closeness yüksek olan oyun ne demek? …
- Ölçüler birbiriyle neden uyumlu ya da uyumsuz (Spearman tablosu)? …

## 5. Topluluk tespiti: Louvain ve Leiden (10 farklı tohum)

In [ ]:
import igraph as ig, leidenalg

nodes = list(G.nodes()); idx = {n: i for i, n in enumerate(nodes)}
edge_list, wts = [], []
for u, v, d in G.edges(data=True):
    edge_list.append((idx[u], idx[v])); wts.append(d["weight"])
g_ig = ig.Graph(n=len(nodes), edges=edge_list)
g_ig.es["weight"] = wts

lou_lab, lou_mod, lei_lab, lei_mod = [], [], [], []
for s in range(10):
    comms = nx.community.louvain_communities(G, weight="weight", seed=s)
    lab = np.zeros(len(nodes), dtype=int)
    for ci, c in enumerate(comms):
        for n in c: lab[idx[n]] = ci
    lou_lab.append(lab)
    lou_mod.append(nx.community.modularity(G, comms, weight="weight"))

    part = leidenalg.find_partition(g_ig, leidenalg.ModularityVertexPartition,
                                    weights="weight", n_iterations=-1, seed=s)
    lei_lab.append(np.array(part.membership)); lei_mod.append(part.modularity)

def pair_scores(labs):
    n_, a_ = [], []
    for i in range(len(labs)):
        for j in range(i + 1, len(labs)):
            n_.append(nmi(labs[i], labs[j])); a_.append(ari(labs[i], labs[j]))
    return np.mean(n_), np.std(n_), np.mean(a_), np.std(a_)

cross_n = [nmi(lou_lab[s], lei_lab[s]) for s in range(10)]
ln, lns, la, las = pair_scores(lou_lab)
en, ens, ea, eas = pair_scores(lei_lab)
topluluk_df = pd.DataFrame([
    {"algoritma": "Louvain", "modülerlik (ort±std)": f"{np.mean(lou_mod):.4f} ± {np.std(lou_mod):.4f}",
     "topluluk sayısı (ort)": np.mean([len(set(l)) for l in lou_lab]),
     "tohumlar arası NMI": f"{ln:.3f} ± {lns:.3f}", "tohumlar arası ARI": f"{la:.3f} ± {las:.3f}"},
    {"algoritma": "Leiden", "modülerlik (ort±std)": f"{np.mean(lei_mod):.4f} ± {np.std(lei_mod):.4f}",
     "topluluk sayısı (ort)": np.mean([len(set(l)) for l in lei_lab]),
     "tohumlar arası NMI": f"{en:.3f} ± {ens:.3f}", "tohumlar arası ARI": f"{ea:.3f} ± {eas:.3f}"},
])
topluluk_df.to_csv(f"{OUT}/topluluk_karsilastirma.csv", index=False, encoding="utf-8-sig")
print(f"Louvain ile Leiden arasındaki NMI (aynı tohum, ortalama): {np.mean(cross_n):.3f}")
topluluk_df

**Yorum:** Modülerlik değeri ne anlama geliyor (0.6 gibi)? İki algoritma benzer sonuç veriyor mu? NMI/ARI yüksekse sonuç kararlı demektir; düşükse topluluk sınırları belirsizdir. …

### Toplulukları adlandır (en büyük 5 topluluk, en iyi Leiden bölümü)

In [ ]:
best = lei_lab[int(np.argmax(lei_mod))]
com_df = pd.DataFrame({"appid": nodes, "topluluk": best})
com_df["oyun"] = [G.nodes[n]["oyun"] for n in nodes]
com_df["degerlendirme"] = [meta.get(n, {}).get("degerlendirme", 0) for n in nodes]
com_df.to_csv(f"{OUT}/topluluk_atamalari.csv", index=False, encoding="utf-8-sig")

gt2 = gt.merge(com_df[["appid", "topluluk"]], on="appid")
gt2["pay"] = gt2["oy"] / gt2.groupby("appid")["oy"].transform("sum")

sizes = com_df["topluluk"].value_counts()
print("Topluluk büyüklükleri (ilk 8):", dict(sizes.head(8)))
for c in sizes.index[:5]:
    sub = gt2[gt2["topluluk"] == c]
    n_g = sub["appid"].nunique()
    top_tags = (sub.groupby("etiket")["pay"].sum() / n_g).sort_values(ascending=False).head(8)
    top_games = com_df[com_df["topluluk"] == c].nlargest(5, "degerlendirme")["oyun"].tolist()
    print(f"\n=== Topluluk {c} ({n_g} oyun) ===")
    print("Baskın etiketler:", ", ".join(f"{t} ({v:.2f})" for t, v in top_tags.items()))
    print("Popüler oyunlar:", "; ".join(top_games))

**Topluluk adları (kendin ver):**
- Topluluk … : "…"
- Topluluk … : "…"
- Topluluk … : "…"
- Topluluk … : "…"
- Topluluk … : "…"

## 6. Null model: ağım rastlantıyla açıklanabilir mi?

Aynı derece dizisine sahip rastgele ağlar (configuration model) üretilir; kümelenme, ortalama yol ve modülerlik gerçek ağla karşılaştırılır. Karşılaştırma **ağırlıksız** ve **en büyük bileşen** üzerinde yapılır.

In [ ]:
N_NULL = 100   # ödev en az 100 istiyor (yaklaşık 5-15 dakika sürebilir)

def olcu_seti(Hh, seed):
    cl = nx.average_clustering(Hh)
    pa, _ = sampled_paths(Hh, k=100, seed=seed)
    comms = nx.community.louvain_communities(Hh, seed=seed)
    mod = nx.community.modularity(Hh, comms)
    return {"kümelenme": cl, "ort_yol": pa, "modülerlik": mod}

gercek = olcu_seti(H, SEED)
degs = [d for _, d in H.degree()]
rows = []
for i in range(N_NULL):
    R = nx.Graph(nx.configuration_model(degs, seed=i))
    R.remove_edges_from(nx.selfloop_edges(R))
    R = lcc(R)
    rows.append(olcu_seti(R, i))
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{N_NULL} rastgele ağ tamam")
null_df = pd.DataFrame(rows)
null_df.to_csv(f"{OUT}/null_model_degerleri.csv", index=False, encoding="utf-8-sig")

ozet = pd.DataFrame({
    "gerçek ağ": pd.Series(gercek),
    "null ortalama": null_df.mean(),
    "null std": null_df.std(),
})
ozet["z-skoru"] = (ozet["gerçek ağ"] - ozet["null ortalama"]) / ozet["null std"]
ozet = ozet.round(4)
ozet.to_csv(f"{OUT}/null_model_ozet.csv", encoding="utf-8-sig")
ozet

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, col in zip(ax, null_df.columns):
    a.hist(null_df[col], bins=20, alpha=0.7)
    a.axvline(gercek[col], color="red", lw=2, label="gerçek ağ")
    a.set_title(col); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/null_model.png", dpi=150); plt.show()

**Yorum:** |z| > 3 ise gerçek ağın değeri rastlantıyla açıklanamaz. Bizim ağda kümelenme ve modülerlik rastgele ağlardan ne kadar yüksek? Bu, oyunların etiket benzerliğine göre gerçekten kümelendiğini söylüyor mu? Null model olmasaydı ne iddia edemezdik? …

## 7. Sonuçları indir

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/sonuclar", "zip", OUT)
files.download("/content/sonuclar.zip")